# BoldSignal — Batch Brain Score Pipeline (Mac Local)
> *Your content, through the brain's eyes — running locally on Apple Silicon.*

Predict human brain engagement for any YouTube video using **Meta's TRIBE v2** fMRI encoding model.
Outputs a **Brain Score (0–100)** across 4 neuroscience dimensions + a second-by-second engagement timeline.

---

## Mac Setup (do this once)
1. Run **Cell 1** (install) — restart the kernel when prompted
2. Set your HuggingFace token before launching Jupyter:
   ```bash
   export HF_TOKEN=REDACTED_HF_TOKEN_token_here
   jupyter notebook
   ```
   - Accept the **VJEPA2** licence gate: huggingface.co/facebook/vjepa2-vitl-fpc64-256
3. Run all remaining cells top-to-bottom

## Mac vs Colab differences
| | Colab | This notebook |
|---|---|---|
| Device | CUDA T4 | Apple MPS (M-series) or CPU |
| Text encoder | Llama-3.2-3B (6 GB) | **Qwen3-0.6B (1.2 GB)** |
| Video encoder | vjepa2-vitg ViT-Giant | **vjepa2-vitl ViT-Large (620 MB)** |
| Audio encoder | w2v-bert-2.0 | w2v-bert-2.0 (unchanged) |
| Storage | Google Drive | `~/BoldSignal/` |
| Install | apt-get + uv | pip + brew |

Each encoder loads → extracts features → is freed from memory before the next one loads.
Peak RAM per extractor: ~1.2 GB. A 16 GB M-series Mac handles this comfortably.

## Local folder layout
```
~/BoldSignal/
├── results.json       ← master results (Brain Score for every video)
├── cache/             ← TRIBE v2 weights + feature cache (~13 GB first run)
├── preds/             ← inference arrays (.npy + .pkl) per video
└── logs/batch.log     ← timestamped run log
```

In [11]:
# ── 1. INSTALL ────────────────────────────────────────────────────────────────
# Run once. Restart the kernel after this cell completes.
import sys
get_ipython().system(f'{sys.executable} -m pip install -q '
    '"tribev2[plotting] @ git+https://github.com/facebookresearch/tribev2.git" '
    'nilearn "scipy>=1.13.0" yt-dlp')

# ffmpeg must be installed via Homebrew (not pip)
import shutil, subprocess
if shutil.which('ffmpeg') is None:
    print('⚠  ffmpeg not found — run in terminal: brew install ffmpeg')
else:
    v = subprocess.check_output(['ffmpeg', '-version']).decode().splitlines()[0]
    print(f'ffmpeg OK: {v}')

print('\nInstall complete — restart the kernel, then run from Cell 2.')


[notice] A new release of pip is available: 23.2.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
ffmpeg OK: ffmpeg version 8.1 Copyright (c) 2000-2026 the FFmpeg developers

Install complete — restart the kernel, then run from Cell 2.


In [12]:
# ── 2. SETUP LOCAL PATHS ──────────────────────────────────────────────────────
import os, json, pickle, time, subprocess, logging
from datetime import datetime
import numpy as np

BASE_DIR     = os.path.expanduser('~/BoldSignal')
CACHE_DIR    = f'{BASE_DIR}/cache'
PREDS_DIR    = f'{BASE_DIR}/preds'
RESULTS_PATH = f'{BASE_DIR}/results.json'
LOG_PATH     = f'{BASE_DIR}/logs/batch.log'

for d in [CACHE_DIR, PREDS_DIR, f'{BASE_DIR}/logs']:
    os.makedirs(d, exist_ok=True)

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s  %(message)s',
    handlers=[logging.FileHandler(LOG_PATH), logging.StreamHandler()]
)
log = logging.getLogger()
print(f'Paths ready — {BASE_DIR}')

Paths ready — /Users/deep/BoldSignal


In [13]:
# ── 3. IMPORTS & DEVICE DETECTION ─────────────────────────────────────────────
import matplotlib.pyplot as plt
from scipy.stats import zscore, pearsonr
from nilearn import datasets, surface
import torch

# MPS = Apple Silicon GPU (M1/M2/M3/M4)
if torch.cuda.is_available():
    DEVICE = 'cuda'
elif torch.backends.mps.is_available():
    DEVICE = 'mps'
else:
    DEVICE = 'cpu'

print(f'Device: {DEVICE}')
if DEVICE == 'mps':
    print('Apple Silicon detected — encoders will use MPS for feature extraction')
elif DEVICE == 'cpu':
    print('CPU only — inference will be slower but functional')

Device: mps
Apple Silicon detected — encoders will use MPS for feature extraction


In [14]:
# ── 4. HF AUTHENTICATION ──────────────────────────────────────────────────────
# Option A (recommended): set HF_TOKEN in your shell before launching Jupyter
#   export HF_TOKEN=REDACTED_HF_TOKEN_token_here
# Option B: run `huggingface-cli login` in your terminal once

from huggingface_hub import login, whoami

REDACTED_HF_TOKEN = 'REDACTED_HF_TOKEN'
# REDACTED_HF_TOKEN = userdata.get('HF_TOKEN')
if REDACTED_HF_TOKEN:
    login(token=REDACTED_HF_TOKEN, add_to_git_credential=False)
    print(f'HF auth OK (from env) — logged in as: {whoami()["name"]}')
else:
    try:
        print(f'HF auth OK (cached token) — logged in as: {whoami()["name"]}')
    except Exception:
        print('⚠  Not authenticated — run `huggingface-cli login` in terminal')
        print('   Required for: facebook/vjepa2-vitl-fpc64-256 (gated model)')

2026-04-19 22:39:14,036  HTTP Request: GET https://huggingface.co/api/whoami-v2 "HTTP/1.1 200 OK"
2026-04-19 22:39:14,091  HTTP Request: GET https://huggingface.co/api/whoami-v2 "HTTP/1.1 200 OK"


HF auth OK (from env) — logged in as: Deep2325


In [20]:
# ── 5. LOAD TRIBE v2 (Mac config) ─────────────────────────────────────────────
from tribev2 import TribeModel
import inspect

log.info('Loading TRIBE v2 — first run downloads ~13 GB to %s', CACHE_DIR)

# Detect which parameters this installed version of from_pretrained supports
_sig = inspect.signature(TribeModel.from_pretrained).parameters
print(f'from_pretrained supports: {list(_sig.keys())}')

_kwargs = {'cache_folder': CACHE_DIR}

if 'cluster' in _sig:
    _kwargs['cluster'] = None           # run locally, no Slurm

if 'device' in _sig:
    _kwargs['device'] = DEVICE

if 'config_update' in _sig:
    _kwargs['config_update'] = {
        # Lighter text encoder (Qwen3-0.6B vs Llama-3.2-3B)
        'data.text_feature.model_name': 'Qwen/Qwen3-0.6B',
        'data.text_feature.layers':     2 / 3,
        # Lighter video encoder (ViT-Large vs ViT-Giant)
        'data.video_feature.image.model_name': 'facebook/vjepa2-vitl-fpc64-256',
        'data.video_feature.image.layers':     2 / 3,
    }
    print('Mac config applied: Qwen3-0.6B + vjepa2-vitl')
else:
    print('⚠  config_update not supported in this tribev2 version — using default encoders')
    print('   Default encoders: Llama-3.2-3B (~6 GB) + vjepa2-vitg (~2 GB)')
    print('   If you hit OOM, upgrade: pip install -U "tribev2 @ git+https://github.com/facebookresearch/tribev2.git"')

model = TribeModel.from_pretrained('facebook/tribev2', **_kwargs)

# Ensure brain encoder is on the right device (fallback if device param not supported)
if hasattr(model, '_model') and model._model is not None:
    model._model = model._model.to(DEVICE)

log.info('Model ready — brain encoder on %s', DEVICE)

2026-04-19 22:40:38,846  Loading TRIBE v2 — first run downloads ~13 GB to /Users/deep/BoldSignal/cache
2026-04-19 22:40:38,972  HTTP Request: HEAD https://huggingface.co/facebook/tribev2/resolve/main/config.yaml "HTTP/1.1 307 Temporary Redirect"
2026-04-19 22:40:38,992  HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/facebook/tribev2/f894e783020944dcd96e5568550afe2aa9743f9f/config.yaml "HTTP/1.1 200 OK"


from_pretrained supports: ['checkpoint_dir', 'checkpoint_name', 'cache_folder', 'cluster', 'device', 'config_update']
Mac config applied: Qwen3-0.6B + vjepa2-vitl


2026-04-19 22:40:39,080  HTTP Request: HEAD https://huggingface.co/facebook/tribev2/resolve/main/best.ckpt "HTTP/1.1 302 Found"
/Users/deep/.pyenv/versions/3.11.7/lib/python3.11/site-packages/neuralset/extractors/base.py:707: UserWarning: LabelEncoder: event_types has not been set, are you sure you want to apply this extractor to all events?
  warnings.warn(
2026-04-19 22:40:39 - WARNING - neuralset.extractors.base:798 - Missing events will be encoded using the default all-zero value (for example, 0 or a zero vector/tensor), which may be indistinguishable from a valid class if that class is also mapped to zeros. Set treat_missing_as_separate_class=True to avoid this.
2026-04-19 22:40:39,105  Missing events will be encoded using the default all-zero value (for example, 0 or a zero vector/tensor), which may be indistinguishable from a valid class if that class is also mapped to zeros. Set treat_missing_as_separate_class=True to avoid this.
INFO - Loading model from /Users/deep/.cache/hug

In [21]:
# ── 6. ATLAS & ROI MAP ───────────────────────────────────────────────────────
destrieux = datasets.fetch_atlas_destrieux_2009()
fsavg5    = datasets.fetch_surf_fsaverage(mesh='fsaverage5')

if 'map_left' in destrieux and 'map_right' in destrieux:
    lh_labels = surface.load_surf_data(destrieux['map_left'])
    rh_labels = surface.load_surf_data(destrieux['map_right'])
else:
    lh_labels = surface.vol_to_surf(
        destrieux['maps'], fsavg5['pial_left'],
        interpolation='nearest_most_frequent').astype(int)
    rh_labels = surface.vol_to_surf(
        destrieux['maps'], fsavg5['pial_right'],
        interpolation='nearest_most_frequent').astype(int)

all_labels  = np.concatenate([lh_labels, rh_labels])
raw         = destrieux['labels']
items       = raw.tolist() if hasattr(raw, 'tolist') else list(raw)
label_names = [v.decode() if isinstance(v, bytes) else str(v) for v in items]

def get_idx(keywords):
    idx = []
    for i, name in enumerate(label_names):
        if any(k.lower() in name.lower() for k in keywords):
            idx.extend(np.where(all_labels == i)[0].tolist())
    return np.array(idx)

ROI = {
    # Default Mode Network — PCC, mPFC, angular gyrus
    'DMN':  get_idx(['cingul', 'front_sup', 'precuneus', 'angular']),

    # Emotional memory circuit
    # Amygdala and hippocampus are subcortical — absent from Destrieux surface atlas.
    # Proxies used:
    #   AMY → temporal pole (G_temporal_pole): anterior temporal cortex directly
    #         adjacent to amygdala; distinct label, no overlap with PHG/HC
    #   PHG → parahippocampal gyrus (G_oc-temp_med-Parahip)
    #   HC  → lingual gyrus (G_oc-temp_med-Lingual): posterior hippocampal projection
    # Note: 'oc-temp_med' is a parent label covering both parahip AND lingual —
    # use the specific child keywords to keep regions separate.
    'AMY':  get_idx(['temporal_pole']),
    'PHG':  get_idx(['parahip']),
    'HC':   get_idx(['oc-temp_med-Lingual']),

    # Social processing
    'pSTS': get_idx(['temp_sup']),
    'FFG':  get_idx(['fusifor']),
    'rTPJ': get_idx(['supramarginal', 'angular']),

    # Sensory salience
    'V1':   get_idx(['cuneus', 'calcarine']),
    'A1':   get_idx(['T_transv', 'temporal_transverse']),
    'INS':  get_idx(['insul', 'Ins']),
}
for k, v in ROI.items():
    print(f'{k:6s}: {len(v):5d} voxels')

# Verify no overlaps in emotional memory circuit
amy_s, phg_s, hc_s = set(ROI['AMY'].tolist()), set(ROI['PHG'].tolist()), set(ROI['HC'].tolist())
print(f'\nOverlap check — AMY∩PHG: {len(amy_s&phg_s)}  AMY∩HC: {len(amy_s&hc_s)}  PHG∩HC: {len(phg_s&hc_s)}  (all should be 0)')

[fetch_atlas_destrieux_2009] Dataset found in /Users/deep/nilearn_data/destrieux_2009

/var/folders/zk/vg8chxss4gl0q6hrm694dpvm0000gn/T/ipykernel_38406/4195490822.py:2: UserWarning: 
The following regions are present in the atlas look-up table,
but missing from the atlas image:

 index          name
    42 L Medial_wall
   117 R Medial_wall

  destrieux = datasets.fetch_atlas_destrieux_2009()


DMN   :  3579 voxels
AMY   :     0 voxels
PHG   :   292 voxels
HC    :   309 voxels
pSTS  :   653 voxels
FFG   :   223 voxels
rTPJ  :   441 voxels
V1    :  1002 voxels
A1    :   245 voxels
INS   :   838 voxels

Overlap check — AMY∩PHG: 0  AMY∩HC: 0  PHG∩HC: 0  (all should be 0)


In [23]:
# ── 7. HELPERS & SCORING ─────────────────────────────────────────────────────

# ── Calibrated normalisation bounds (pilot: 5 YouTube Shorts, 2026-04-07) ─────
# Derived from DIAG 1 observed ranges ± 20% (5-video pilot, all YouTube Shorts).
# Re-run DIAG 1 with diverse content types to expand these bounds over time.
ATTN_RAW_MIN, ATTN_RAW_MAX =  0.377,  0.486
MEMO_RAW_MIN, MEMO_RAW_MAX = -0.144,  0.153
SOCI_RAW_MIN, SOCI_RAW_MAX = -0.083,  0.607
HOOK_RAW_MIN, HOOK_RAW_MAX =  0.024,  0.082

def normalise(x, in_min=None, in_max=None, out_min=0.0, out_max=1.0):
    if in_min is None: in_min = float(np.min(x))
    if in_max is None: in_max = float(np.max(x))
    x = np.clip(x, in_min, in_max)
    return (x - in_min) / (in_max - in_min + 1e-8) * (out_max - out_min) + out_min

def roi_mean(preds, idx):
    if len(idx) == 0: return np.zeros(preds.shape[0])
    valid = idx[idx < preds.shape[1]]
    return preds[:, valid].mean(axis=1)

def safe_zscore(x):
    if len(x) == 0 or x.std() == 0: return np.zeros(max(len(x), 1))
    return zscore(x)

def get_times(segments, n):
    try:    return np.array([s.start for s in segments])
    except AttributeError:
        try: return np.array([s['onset'] for s in segments])
        except: return np.arange(n, dtype=float)

def detect_narrative_mode(segments, override=None):
    """Return 'narrative' or 'non_narrative'.

    TRIBE v2 segments do not carry transcript data, so auto-detection is not
    yet possible. Always returns 'non_narrative' until Whisper ASR is integrated.
    Pass override='narrative' or override='non_narrative' to force a mode.
    """
    if override is not None:
        return override
    return 'non_narrative'

def construct_narrative_template(n):
    return np.exp(-4 * (np.linspace(0, 1, n) - 0.8)**2)

def score_sustained_attention(preds, mode):
    DMN = roi_mean(preds, ROI['DMN'])
    if len(DMN) == 0: return 0.0
    if mode == 'non_narrative':
        raw = 0.40*(-DMN.mean()) + 0.60*(1-(DMN.std()/(np.ptp(DMN)+1e-8)))
        return float(normalise(raw, ATTN_RAW_MIN, ATTN_RAW_MAX) * 30)
    r = pearsonr(DMN, construct_narrative_template(len(DMN)))[0] if len(DMN) > 2 else 0.0
    return float(normalise(r, -1.0, 1.0) * 30)

def score_emotional_memory(preds):
    AMY = safe_zscore(roi_mean(preds, ROI['AMY']))
    PHG = safe_zscore(roi_mean(preds, ROI['PHG']))
    HC  = safe_zscore(roi_mean(preds, ROI['HC']))
    n   = len(AMY)
    w   = np.ones(n); w[int(0.8*n):] = 2.0; w /= w.sum()
    c   = 0.35*np.average(AMY,weights=w) + 0.45*np.average(PHG,weights=w) + 0.20*np.average(HC,weights=w)
    return float(normalise(c, MEMO_RAW_MIN, MEMO_RAW_MAX) * 25)

def score_social_processing(preds):
    # Measure each ROI's activation relative to the whole-brain mean at each timepoint.
    # z-score(timeseries).mean() is always 0 by definition — instead we compare
    # the ROI mean against the instantaneous whole-brain mean and std.
    whole_mean = preds.mean(axis=1)       # (n_timepoints,)
    whole_std  = preds.std(axis=1) + 1e-8
    def szm(k):
        roi_ts = roi_mean(preds, ROI[k])  # (n_timepoints,)
        return float(((roi_ts - whole_mean) / whole_std).mean())
    s = 0.45*szm('pSTS') + 0.25*szm('FFG') + 0.30*szm('rTPJ')
    return float(normalise(s, SOCI_RAW_MIN, SOCI_RAW_MAX) * 25)

def score_sensory_salience(preds, segments):
    times = get_times(segments, len(preds))
    def wm(mask, k):
        sub = preds[mask] if mask.sum() > 0 else preds[:1]
        return float(roi_mean(sub, ROI[k]).mean())
    me, mm, mr = times<=5, (times>5)&(times<=15), times>15
    hook = (0.60*(0.40*wm(me,'V1')+0.30*wm(me,'A1')+0.30*wm(me,'INS'))
           +0.30*(0.50*wm(mm,'V1')+0.50*wm(mm,'INS'))
           +0.10*np.mean([wm(mr,'V1'),wm(mr,'A1'),wm(mr,'INS')]))
    return float(normalise(hook, HOOK_RAW_MIN, HOOK_RAW_MAX) * 20)

def compute_isc_proxy(preds):
    # Returns ISC multiplier in [0.70, 1.15] directly.
    # Lower voxel-wise variance = more consistent brain response = higher ISC.
    # np.clip requires a_min <= a_max, so we handle the inverted mapping explicitly.
    # Pilot range: 0.004 (consistent) → 1.15, 0.024 (variable) → 0.70
    vov = float(np.clip(preds.var(axis=0).mean(), 0.004, 0.024))
    t   = (vov - 0.004) / (0.024 - 0.004)  # 0.0 (consistent) → 1.0 (variable)
    return 1.15 - t * 0.45                  # 1.15 → 0.70

def compute_brain_score(preds, segments):
    mode     = detect_narrative_mode(segments)
    isc_mult = compute_isc_proxy(preds)   # already in [0.70, 1.15]
    d1 = score_sustained_attention(preds, mode)
    d2 = score_emotional_memory(preds)
    d3 = score_social_processing(preds)
    d4 = score_sensory_salience(preds, segments)
    return {
        'brain_score':    min(round((d1+d2+d3+d4)*isc_mult, 1), 100.0),
        'isc_multiplier': round(isc_mult, 3),
        'content_mode':   mode,
        'dimensions':     {
            'sustained_attention': round(d1, 2),
            'emotional_memory':    round(d2, 2),
            'social_processing':   round(d3, 2),
            'sensory_salience':    round(d4, 2),
        },
    }

def build_engagement_timeline(preds, segments):
    times     = get_times(segments, len(preds))
    attention = normalise(-roi_mean(preds, ROI['DMN']))
    emotional = normalise((roi_mean(preds,ROI['AMY'])+roi_mean(preds,ROI['PHG']))/2)
    return {'times':times.tolist(), 'attention':attention.tolist(), 'emotional':emotional.tolist()}

print('All functions defined ✓')

All functions defined ✓


In [24]:
# ── 8. VIDEO LIST — add your own URLs here ───────────────────────────────────
# Supports any YouTube URL (shorts, standard, etc.)
# Commented-out URLs are skipped. Re-run this cell to update the list.

VIDEO_URLS = [
    # "https://youtube.com/shorts/QIH5XV3Vays?si=srSTdRm9E8kkZUlb",
    "https://youtube.com/shorts/VqBmPmVxwV8?si=6hiaYLdtyEfNRuo5",
    # "https://youtube.com/shorts/3D9uIn2oBLI?si=FrZ64srudTe3ajln",
    # "https://youtube.com/shorts/VfR1b_t13oA?si=65Bd4c8x4Ag1ZWd4",
    "https://youtube.com/shorts/IQxea9UB1nQ?si=UT58Un96hwoj0Qan",
    # "https://youtube.com/shorts/b2EUuXOF2kc?si=yn3CSBfKu_skIUBl",
    # "https://youtube.com/shorts/1ntwS9EbaQ8?si=ILLC-HHJSzzeGLaW",
    # "https://youtube.com/shorts/cZUU2gwqLeA?si=MNBYSnQgXQ8FMIpd",
    "https://youtube.com/shorts/JktNgjnAv9s?si=jVBTjOOyRRTcVQjq",
    # "https://youtube.com/shorts/ChZH0eyjPe0?si=inCu3CQTTeWBmDpM",
    # "https://youtube.com/shorts/XpUBK_zw-q0?si=dyQkBWuWviTHe1cl",
    "https://youtube.com/shorts/FhEcR9sTvqI?si=pJq_depDmaJgAVXD",
    "https://youtube.com/shorts/BgqyapYlwy8?si=kbo6FqioKf02H1De",
    # "https://youtube.com/shorts/Wfm-VzQIQsI?si=oa4wFFXbuAFxDhom",
]

TARGET_FPS = 4   # 4 fps ≈ 6x faster inference vs native 24 fps — sufficient for TRIBE v2


In [26]:
# ── 9. BATCH LOOP ─────────────────────────────────────────────────────────────
# - Skips videos already scored (reads ~/BoldSignal/results.json)
# - Caches inference arrays locally — safe to interrupt and resume
# - Temp files go to /tmp (not /content — that's Colab only)

_ns = globals()
_missing = [f for f in ['compute_brain_score', 'ROI', 'model'] if f not in _ns]
if _missing:
    raise RuntimeError(
        f'Missing: {_missing}\n'
        'Re-run cells 2–7 (skip cell 1, packages already installed).'
    )

def _video_id(url):
    import re, hashlib
    m = re.search(r'(?:v=|shorts/|youtu\.be/)([\w-]+)', url)
    return m.group(1) if m else hashlib.md5(url.encode()).hexdigest()[:12]

def _preds_paths(vid_id):
    return f'{PREDS_DIR}/{vid_id}_preds.npy', f'{PREDS_DIR}/{vid_id}_segments.pkl'

def _download(url, proc_path):
    raw = proc_path.replace('.mp4', '_raw.mp4')
    r = subprocess.run([
        'yt-dlp', '--extractor-args', 'youtube:player_client=android,web',
        '--format', 'bestvideo[ext=mp4][height<=480]+bestaudio[ext=m4a]/best[ext=mp4]/best',
        '--merge-output-format', 'mp4', '--output', raw, url,
    ], capture_output=True, text=True)
    if not os.path.exists(raw):
        raise RuntimeError(f'Download failed: {r.stderr[-200:]}')
    subprocess.run([
        'ffmpeg', '-y', '-i', raw, '-vf', f'fps={TARGET_FPS}',
        '-c:v', 'libx264', '-crf', '23', '-c:a', 'aac', proc_path
    ], capture_output=True)
    os.remove(raw)
    if not os.path.exists(proc_path):
        raise RuntimeError('ffmpeg failed')

def _load_results():
    if os.path.exists(RESULTS_PATH):
        with open(RESULTS_PATH) as f: return json.load(f)
    return []

def _save_results(results):
    with open(RESULTS_PATH, 'w') as f: json.dump(results, f, indent=2)

all_results = _load_results()
done_ids    = {r['video_id'] for r in all_results if r.get('brain_score') is not None}
total       = len(VIDEO_URLS)
log.info(f'Batch start: {total} URLs, {len(done_ids)} already scored')

for i, url in enumerate(VIDEO_URLS):
    vid_id = _video_id(url)
    preds_path, segs_path = _preds_paths(vid_id)

    if vid_id in done_ids:
        log.info(f'[{i+1}/{total}] skip (already scored): {vid_id}')
        continue

    log.info(f'[{i+1}/{total}] {url}  id={vid_id}')
    t0 = time.time()
    entry = {
        'url': url, 'video_id': vid_id,
        'timestamp': datetime.now().isoformat(),
        'brain_score': None, 'isc_multiplier': None,
        'content_mode': None, 'dimensions': None,
        'error': None, 'processing_time_s': None,
        'inference_cached': False,
        'device': DEVICE,
    }

    try:
        if os.path.exists(preds_path) and os.path.exists(segs_path):
            log.info('  loading cached preds...')
            preds_i = np.load(preds_path)
            with open(segs_path, 'rb') as f: segs_i = pickle.load(f)
            entry['inference_cached'] = True
        else:
            proc = f'/tmp/boldsignal_{vid_id}.mp4'
            log.info('  downloading + preprocessing...')
            _download(url, proc)
            log.info(f'  {os.path.getsize(proc)/1e6:.1f} MB at {TARGET_FPS} fps')

            log.info('  running TRIBE v2 inference...')
            df_i = model.get_events_dataframe(video_path=proc)
            preds_i, segs_i = model.predict(events=df_i)
            log.info(f'  preds shape: {preds_i.shape}')

            np.save(preds_path, preds_i)
            with open(segs_path, 'wb') as f: pickle.dump(segs_i, f)
            log.info('  preds saved ✓')
            os.remove(proc)

        r = compute_brain_score(preds_i, segs_i)
        entry.update(r)
        log.info(f'  brain score: {r["brain_score"]}/100')

    except Exception as e:
        entry['error'] = str(e)
        log.error(f'  ERROR: {e}')

    finally:
        for p in [f'/tmp/boldsignal_{vid_id}.mp4', f'/tmp/boldsignal_{vid_id}_raw.mp4']:
            if os.path.exists(p): os.remove(p)
        entry['processing_time_s'] = round(time.time() - t0, 1)

    all_results.append(entry)
    if entry['brain_score'] is not None:
        done_ids.add(vid_id)
    _save_results(all_results)
    log.info(f'  saved ✓  elapsed: {entry["processing_time_s"]}s')

ok = [r for r in all_results if r['brain_score'] is not None]
log.info(f'Complete: {len(ok)}/{len(all_results)} scored')
print(f"\n{'Score':>6}  {'Mode':<15}  {'Device':<6}  {'ID':<14}  URL")
print('─' * 90)
for r in sorted(ok, key=lambda x: x['brain_score'], reverse=True):
    dev = r.get('device', '?')
    print(f"{r['brain_score']:>6.1f}  {r['content_mode']:<15}  {dev:<6}  {r['video_id']:<14}  {r['url']}")

2026-04-19 22:45:33,132  Batch start: 5 URLs, 0 already scored
2026-04-19 22:45:33,134  [1/5] https://youtube.com/shorts/VqBmPmVxwV8?si=6hiaYLdtyEfNRuo5  id=VqBmPmVxwV8
2026-04-19 22:45:33,135    downloading + preprocessing...
2026-04-19 22:45:36,425    2.4 MB at 4 fps
2026-04-19 22:45:36,426    running TRIBE v2 inference...
Extract audio from video events:   0%|          | 0/1 [00:00<?, ?it/s]

MoviePy - Writing audio in /tmp/boldsignal_VqBmPmVxwV8.wav


Extract audio from video events: 100%|██████████| 1/1 [00:00<00:00,  7.53it/s]
/Users/deep/.pyenv/versions/3.11.7/lib/python3.11/site-packages/neuralset/events/utils.py:134: UserWarning: The events dataframe contains an `Index` column. This is dangerous, please add drop=True in calls to df.reset_index(). Dropping it automatically.
  warnings.warn(msg)


MoviePy - Done.


Extracting words from audio:   0%|          | 0/1 [00:55<?, ?it/s]


KeyboardInterrupt: 

In [ ]:
# ── RE-SCORE: clear brain_score cache, keep preds ────────────────────────────
# Run after updating calibration bounds in cell 7.

results_reset = []
for r in _load_results():
    r_reset = dict(r)
    r_reset['brain_score']    = None
    r_reset['dimensions']     = None
    r_reset['isc_multiplier'] = None
    r_reset['content_mode']   = None
    results_reset.append(r_reset)
_save_results(results_reset)
print(f'Reset {len(results_reset)} entries — preds intact, scores cleared.')
print('Now re-run the batch loop cell (cell 9) to rescore from cache.')

NameError: name '_load_results' is not defined

In [ ]:
# ── 10. VISUALISE A RESULT ────────────────────────────────────────────────────
import matplotlib.pyplot as plt

BG     = '#0D1117'
PANEL  = '#161B22'
BLUE   = '#4FC3F7'
PURPLE = '#CE93D8'
RED    = '#FF6B6B'
GRID   = '#21262D'
TEXT   = '#C9D1D9'
SUBTEXT= '#8B949E'

VIDEO_ID = _video_id(VIDEO_URLS[0])   # ← change index to pick a different video

preds_path, segs_path = _preds_paths(VIDEO_ID)
preds_v = np.load(preds_path)
with open(segs_path, 'rb') as f: segs_v = pickle.load(f)

cached = next((x for x in all_results if x['video_id'] == VIDEO_ID), None)
if cached is None or cached.get('dimensions') is None:
    r = compute_brain_score(preds_v, segs_v)
    r['url'] = VIDEO_ID
else:
    r = cached

timeline  = build_engagement_timeline(preds_v, segs_v)
times     = np.array(timeline['times'])
attention = np.array(timeline['attention'])
emotional = np.array(timeline['emotional'])

def _style_ax(ax):
    ax.set_facecolor(PANEL)
    ax.tick_params(colors=SUBTEXT, labelsize=10)
    ax.yaxis.label.set_color(TEXT)
    ax.xaxis.label.set_color(TEXT)
    for spine in ax.spines.values():
        spine.set_edgecolor(GRID)
    ax.grid(color=GRID, linewidth=0.6, alpha=0.8)

# ── Figure 1: Engagement Timeline ────────────────────────────────────────────
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 6), sharex=True,
                                facecolor=BG, gridspec_kw={'hspace': 0.08})
fig.suptitle(
    f"Brain Score: {r['brain_score']}/100  ·  {r['url'][:70]}",
    color=TEXT, fontsize=14, fontweight='bold', x=0.02, ha='left', y=0.98
)

for ax, signal, color, label in [
    (ax1, attention, BLUE,   'Attention'),
    (ax2, emotional, PURPLE, 'Emotional'),
]:
    _style_ax(ax)
    ax.fill_between(times, signal, alpha=0.18, color=color)
    ax.plot(times, signal, color=color, lw=2, label=label)
    ax.set_ylabel(label, color=TEXT, fontsize=11)
    ax.set_ylim(0, 1)
    ax.set_xlim(times[0], times[-1])

ax1.axhline(0.4, color=RED, ls='--', lw=1.2, alpha=0.7)
ax1.text(times[-1]*0.98, 0.42, 'drop-off', color=RED, fontsize=9, ha='right', alpha=0.85)

in_drop, seg_start = False, None
for i, (t, d) in enumerate(zip(times, attention < 0.4)):
    if d and not in_drop:
        seg_start = t; in_drop = True
    elif not d and in_drop:
        ax1.axvspan(seg_start, t, color=RED, alpha=0.07)
        in_drop = False
if in_drop:
    ax1.axvspan(seg_start, float(times[-1]), color=RED, alpha=0.07)

ax2.set_xlabel('Time (s)', color=TEXT, fontsize=11)
for ax, lbl in [(ax1, 'ATTENTION SIGNAL'), (ax2, 'EMOTIONAL SIGNAL')]:
    ax.text(0.01, 0.92, lbl, transform=ax.transAxes,
            color=SUBTEXT, fontsize=8, fontweight='bold', alpha=0.7)

fig.subplots_adjust(top=0.93, hspace=0.08)
plt.show()

# ── Figure 2: Score Breakdown (Radar) ────────────────────────────────────────
dims       = r['dimensions']
dim_keys   = ['sustained_attention','emotional_memory',
              'social_processing','sensory_salience']
dim_labels = ['Attention\n(30)', 'Memory\n(25)', 'Social\n(25)', 'Salience\n(20)']
dim_max    = [30, 25, 25, 20]
dim_scores = [dims[k] for k in dim_keys]
pct        = [s / m * 100 for s, m in zip(dim_scores, dim_max)]
dot_colors = [BLUE, PURPLE, '#66BB6A', '#FF8A65']

N      = len(dim_labels)
angles = [n / N * 2 * np.pi for n in range(N)] + [0]
pct_c  = pct + [pct[0]]

fig2 = plt.figure(figsize=(9, 7), facecolor=BG)
ax = fig2.add_subplot(111, polar=True, facecolor=PANEL)

for ring in [25, 50, 75, 100]:
    ax.plot(angles, [ring] * (N + 1), color=GRID, lw=0.8, alpha=0.6)
for angle in angles[:-1]:
    ax.plot([angle, angle], [0, 100], color=GRID, lw=0.8, alpha=0.5)

ax.fill(angles, pct_c, color=BLUE, alpha=0.15)
ax.plot(angles, pct_c, color=BLUE, lw=2.5)

for angle, p, s, m, c in zip(angles[:-1], pct, dim_scores, dim_max, dot_colors):
    ax.scatter(angle, p, color=c, s=80, zorder=5)
    ax.text(angle, p + 18, f'{s:.1f}/{m}',
            ha='center', va='center', fontsize=10, color=c, fontweight='bold')

ax.set_xticks(angles[:-1])
ax.set_xticklabels(dim_labels, color=TEXT, fontsize=11)
ax.set_yticks([25, 50, 75, 100])
ax.set_yticklabels(['25%', '50%', '75%', '100%'], color=SUBTEXT, fontsize=8)
ax.set_ylim(0, 125)
ax.spines['polar'].set_color(GRID)
ax.tick_params(colors=SUBTEXT)

fig2.suptitle(
    f"Score Breakdown — {r['brain_score']}/100  ·  ISC ×{r['isc_multiplier']}",
    color=TEXT, fontsize=14, fontweight='bold', y=0.98
)
fig2.subplots_adjust(top=0.88)
plt.show()